In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U "protobuf<4.21"

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.model_selection import train_test_split
import gc

# --- CONFIGURATION ---
DATA_DIR = '/kaggle/input/grand-xray-slam-division-b' 
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train2')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test2')
TRAIN_CSV = os.path.join(DATA_DIR, 'train2.csv')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission_2.csv')

WORK_DIR = '/kaggle/working'
BACKUP_DIR = os.path.join(WORK_DIR, 'backup_checkpoint')
LOG_FILE = os.path.join(WORK_DIR, 'training_log.csv')
BEST_MODEL_PATH = os.path.join(WORK_DIR, 'best_model.keras')

IMAGE_SIZE = [320, 320]
EPOCHS = 12
LEARNING_RATE = 1e-4 

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 
    'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 
    'Pneumonia', 'Pneumothorax', 'Support Devices'
]

def setup_hardware():
    # FORCE Single GPU Setup
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            # Only use the first GPU (index 0)
            tf.config.set_visible_devices(gpus[0], 'GPU')
            logical_gpus = tf.config.list_logical_devices('GPU')
            print(f"✅ Running on Single GPU: {logical_gpus[0].name}")
        except RuntimeError as e:
            print(e)
    else:
        print("⚠️ No GPU detected, running on CPU")
        
    # No MirroredStrategy needed for single GPU
    return tf.distribute.get_strategy()

def process_img(file_path, label=None):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    if label is not None:
        return img, label
    return img

def augment_img(img, label):
    img = tf.image.random_flip_left_right(img)
    return img, label

def validate_and_clean_data(df):
    # FAST SCAN: Checks existence and non-zero size only
    print("🔍 Fast scanning image files...")
    valid_rows = []
    for index, row in df.iterrows():
        path = row['Image_path']
        # Check if file exists AND size > 0 (Avoids DecodeJpeg crash)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            valid_rows.append(index)
            
    print(f"✅ Scan complete. Kept {len(valid_rows)}/{len(df)} images.")
    return df.loc[valid_rows].reset_index(drop=True)

def get_dataset(df, batch_size, is_train=True):
    file_paths = df['Image_path'].values
    AUTOTUNE = tf.data.AUTOTUNE
    
    labels = df[LABELS].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(process_img, num_parallel_calls=AUTOTUNE)
    
    if is_train:
        ds = ds.shuffle(2048)
        ds = ds.map(augment_img, num_parallel_calls=AUTOTUNE)
    
    # Standard batching (Drop remainder not strictly needed on Single GPU, but good for shape consistency)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def build_model():
    base_model = tf.keras.applications.DenseNet121(
        weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3)
    )
    base_model.trainable = True 
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = base_model(inputs)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    
    # Linear output for stability
    outputs = layers.Dense(len(LABELS), activation='linear', dtype='float32')(x)
    
    return tf.keras.Model(inputs, outputs)

def main():
    strategy = setup_hardware()
    
    # Single GPU Batch Size
    # 48 is safe for T4 GPU at 320x320 resolution
    GLOBAL_BATCH_SIZE = 48
    print(f"🎯 Global Batch Size: {GLOBAL_BATCH_SIZE}")

    print("Loading dataframe...")
    try:
        df = pd.read_csv(TRAIN_CSV)
        df['Image_path'] = df['Image_name'].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x))
        df[LABELS] = df[LABELS].fillna(0)
        
        # Subsample 60%
        print(f"Original size: {len(df)}")
        df = df.sample(frac=0.6, random_state=42).reset_index(drop=True)
        
        # Run Fast Scan
        df = validate_and_clean_data(df)
        
        train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return

    gc.collect()

    # No scope needed for single GPU, but keeping it doesn't hurt
    with strategy.scope():
        model = build_model()
        
        # High Stability Optimizer Settings
        optimizer = optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0, epsilon=1e-4)
        loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
        
        model.compile(
            optimizer=optimizer,
            loss=loss_fn,
            metrics=[tf.keras.metrics.AUC(multi_label=True, name='auc', from_logits=True)]
        )
        
    my_callbacks = [
        callbacks.ModelCheckpoint(BEST_MODEL_PATH, save_best_only=True, monitor='val_auc', mode='max', verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=2, verbose=1),
        callbacks.BackupAndRestore(backup_dir=BACKUP_DIR),
        callbacks.CSVLogger(LOG_FILE, append=True),
        callbacks.TerminateOnNaN() 
    ]

    print("Starting training...")
    try:
        history = model.fit(
            get_dataset(train_df, GLOBAL_BATCH_SIZE, is_train=True),
            epochs=EPOCHS,
            validation_data=get_dataset(val_df, GLOBAL_BATCH_SIZE, is_train=False),
            callbacks=my_callbacks
        )
    except KeyboardInterrupt:
        print("\n⚠️ Training interrupted.")
        model.save('interrupted_model.keras')
    except Exception as e:
        print(f"❌ Error: {e}")

    print("Generating predictions...")
    try:
        if os.path.exists(BEST_MODEL_PATH):
            model = tf.keras.models.load_model(BEST_MODEL_PATH)
        
        sub_df = pd.read_csv(SAMPLE_SUB)
        test_files = [os.path.join(TEST_IMG_DIR, x) for x in sub_df['Image_name']]
        test_ds = tf.data.Dataset.from_tensor_slices(test_files)
        test_ds = test_ds.map(lambda x: process_img(x, label=None), num_parallel_calls=tf.data.AUTOTUNE)
        test_ds = test_ds.batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        
        logits = model.predict(test_ds, verbose=1)
        probs = tf.nn.sigmoid(logits).numpy()
        
        sub_df[LABELS] = probs
        sub_df.to_csv('submission.csv', index=False)
        print("✅ Submission saved!")
    except Exception as e:
        print(f"❌ Inference Error: {e}")

    if os.path.exists(BACKUP_DIR):
        import shutil
        shutil.rmtree(BACKUP_DIR)

if __name__ == "__main__":
    main()

2025-11-21 17:28:26.177850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763746106.203490     198 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763746106.211204     198 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Running on Single GPU: /device:GPU:0
🎯 Global Batch Size: 48
Loading dataframe...


I0000 00:00:1763746112.105271     198 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Original size: 108494
🔍 Fast scanning image files...
✅ Scan complete. Kept 65093/65096 images.
Starting training...
Epoch 1/12


I0000 00:00:1763746284.974173     248 service.cc:148] XLA service 0x78561c003c30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1763746284.974214     248 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1763746298.697898     248 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1763746408.658062     248 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1220/1220 ━━━━━━━━━━━━━━━━━━━━ 0s 788ms/step - auc: 0.8019 - loss: 0.4012
Epoch 1: val_auc improved from -inf to 0.87888, saving model to /kaggle/working/best_model.keras
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 1362s 911ms/step - auc: 0.8020 - loss: 0.4012 - val_auc: 0.8789 - val_loss: 0.3471 - learning_rate: 1.0000e-04
Epoch 2/12
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 0s 781ms/step - auc: 0.8897 - loss: 0.3017
Epoch 2: val_auc improved from 0.87888 to 0.90572, saving model to /kaggle/working/best_model.keras
1220/1220 ━━━━━━━━━━━━━━━━━━━━ 1123s 900ms/step - auc: 0.8897 - loss: 0.3017 - val_auc: 0.9057 - val_loss: 0.2898 - learning_rate: 1.0000e-04
Epoch 3/12
 254/1220 ━━━━━━━━━━━━━━━━━━━━ 12:38 786ms/step - auc: 0.9030 - loss: 0.2851
⚠️ Training interrupted.
Generating predictions...
 89/999 ━━━━━━━━━━━━━━━━━━━━ 12:46 843ms/step

KeyboardInterrupt: 